# 🏥 Healthcare Analytics — Patient Length of Stay Prediction

> **Goal:** Predict the Length of Stay (LOS) for each patient so that hospitals can optimally allocate resources.

### Pipeline Overview
1. Data Loading & Exploration
2. Data Preparation & Missing Value Imputation
3. Feature Engineering (Count Encoding)
4. Model Training: Naive Bayes | XGBoost | Neural Network
5. Prediction & Results Comparison

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

np.set_printoptions(suppress=True)
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# ----- Load datasets -----
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')

---
## 2. Data Exploration

### 2.1 Overview

In [ ]:
train.head()

In [ ]:
train.info()
print('\nUnique Stay values:', train['Stay'].unique())

### 2.2 Missing Values

In [ ]:
print('=== Train — Missing Values ===')
print(train.isnull().sum().sort_values(ascending=False))

print('\n=== Test — Missing Values ===')
print(test.isnull().sum().sort_values(ascending=False))

### 2.3 Cardinality

In [ ]:
print('=== Train — Unique Values per Column ===')
for col in train.columns:
    print(f'  {col:<40} {train[col].nunique()}')

In [ ]:
print('=== Test — Unique Values per Column ===')
for col in test.columns:
    print(f'  {col:<40} {test[col].nunique()}')

### 2.4 Target Distribution

In [ ]:
stay_order = ['0-10','11-20','21-30','31-40','41-50','51-60',
              '61-70','71-80','81-90','91-100','More than 100 Days']

stay_counts = train['Stay'].value_counts().reindex(stay_order)

plt.figure(figsize=(12, 4))
stay_counts.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Distribution of Patient Length of Stay (Train)')
plt.xlabel('Stay (days)')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 3. Data Preparation

### 3.1 Impute Missing Values

In [ ]:
# Fill missing Bed Grade with mode (most common grade)
for df in [train, test]:
    df['Bed Grade'].fillna(df['Bed Grade'].mode()[0], inplace=True)
    df['City_Code_Patient'].fillna(df['City_Code_Patient'].mode()[0], inplace=True)

print('Missing values after imputation:')
print('Train:', train.isnull().sum().sum())
print('Test: ', test.isnull().sum().sum())

### 3.2 Encode Target Variable

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Ordinal mapping for Stay — preserves natural order
stay_map = {v: i for i, v in enumerate(stay_order)}
train['Stay'] = train['Stay'].map(stay_map)

print('Stay label mapping:')
for k, v in stay_map.items():
    print(f'  {k:<25} → {v}')

### 3.3 Combine Train & Test for Consistent Encoding

In [ ]:
# Add a sentinel Stay value for test rows
test['Stay'] = -1
df = pd.concat([train, test], ignore_index=True)
print(f'Combined dataset shape: {df.shape}')

In [ ]:
# Categorical columns to label-encode
cat_cols = ['Hospital_type_code', 'Hospital_region_code', 'Department',
            'Ward_Type', 'Ward_Facility_Code',
            'Type of Admission', 'Severity of Illness', 'Age']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print('Label encoding done for:', cat_cols)

In [ ]:
# Split back into train / test
train = df[df['Stay'] != -1].copy()
test  = df[df['Stay'] == -1].copy()

print(f'Train: {train.shape} | Test: {test.shape}')

---
## 4. Feature Engineering

In [ ]:
def count_encode(train, test, cols, name):
    """Adds a count-based feature: number of cases per group."""
    temp_tr = train.groupby(cols)['case_id'].count().reset_index().rename(columns={'case_id': name})
    temp_te = test.groupby(cols)['case_id'].count().reset_index().rename(columns={'case_id': name})

    train = train.merge(temp_tr, how='left', on=cols)
    test  = test.merge(temp_te, how='left', on=cols)

    train[name] = train[name].astype(float).fillna(np.median(temp_tr[name]))
    test[name]  = test[name].astype(float).fillna(np.median(temp_te[name]))

    return train, test


train, test = count_encode(train, test, ['patientid'],
                           name='count_id_patient')
train, test = count_encode(train, test, ['patientid', 'Hospital_region_code'],
                           name='count_id_patient_hospitalCode')
train, test = count_encode(train, test, ['patientid', 'Ward_Facility_Code'],
                           name='count_id_patient_wardfacilityCode')

print('Count encoding features added.')
print(f'Train shape: {train.shape} | Test shape: {test.shape}')

In [ ]:
# Drop columns not useful for modelling
DROP_TRAIN = ['case_id', 'patientid', 'Hospital_region_code', 'Ward_Facility_Code']
DROP_TEST  = ['Stay', 'patientid', 'Hospital_region_code', 'Ward_Facility_Code']

train1 = train.drop(DROP_TRAIN, axis=1)
test1  = test.drop(DROP_TEST, axis=1)

print(f'Model train features: {train1.shape} | Model test features: {test1.shape}')

In [ ]:
from sklearn.model_selection import train_test_split

X = train1.drop('Stay', axis=1)
y = train1['Stay']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=100, stratify=y
)

print(f'X_train: {X_train.shape} | X_val: {X_val.shape}')

---
## 5. Models

### 5.1 Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

nb_model = GaussianNB()
nb_model.fit(X_train.values, y_train.values)

pred_nb_val = nb_model.predict(X_val.values)
acc_nb = accuracy_score(y_val, pred_nb_val)

print(f'Naive Bayes — Validation Accuracy: {acc_nb*100:.2f}%')

### 5.2 XGBoost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    max_depth=4,
    learning_rate=0.1,
    n_estimators=800,
    objective='multi:softmax',
    num_class=11,           # explicitly set number of classes
    reg_alpha=0.5,
    reg_lambda=1.5,
    booster='gbtree',
    n_jobs=4,
    min_child_weight=2,
    base_score=0.5,         # corrected: base_score should be in [0,1]
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

pred_xgb_val = xgb_model.predict(X_val)
acc_xgb = accuracy_score(y_val, pred_xgb_val)

print(f'\nXGBoost — Validation Accuracy: {acc_xgb*100:.2f}%')

### 5.3 Neural Network (Keras / TensorFlow)

In [ ]:
from sklearn import preprocessing

# Scale features using the TRAINING set statistics only (no data leakage)
scaler = preprocessing.StandardScaler()

X_full = train.drop(DROP_TRAIN + ['Stay'], axis=1, errors='ignore')
y_full = train['Stay']

X_train_nn, X_val_nn, y_train_nn, y_val_nn = train_test_split(
    X_full, y_full, test_size=0.20, random_state=100, stratify=y_full
)

X_train_nn_sc = scaler.fit_transform(X_train_nn)
X_val_nn_sc   = scaler.transform(X_val_nn)

print(f'NN X_train: {X_train_nn_sc.shape} | X_val: {X_val_nn_sc.shape}')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

NUM_CLASSES = 11
N_FEATURES  = X_train_nn_sc.shape[1]  # dynamically inferred — no hardcoding!

a = to_categorical(y_train_nn, num_classes=NUM_CLASSES)
b = to_categorical(y_val_nn,   num_classes=NUM_CLASSES)

print(f'Input features: {N_FEATURES} | Output classes: {NUM_CLASSES}')

In [ ]:
# Build model — input_shape now correctly uses (N_FEATURES,)
model = Sequential([
    Dense(64,  activation='relu', input_shape=(N_FEATURES,)),
    BatchNormalization(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(512, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',                    # Adam converges faster than plain SGD
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# EarlyStopping prevents overfitting and saves training time
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
tensorboard = tf.keras.callbacks.TensorBoard('logs_keras')

history = model.fit(
    X_train_nn_sc, a,
    epochs=30,
    batch_size=256,
    validation_data=(X_val_nn_sc, b),
    callbacks=[early_stop, tensorboard],
    verbose=1
)

In [ ]:
# Evaluate Neural Network
loss_nn, acc_nn = model.evaluate(X_val_nn_sc, b, verbose=0)
print(f'Neural Network — Validation Accuracy: {acc_nn*100:.2f}% | Loss: {loss_nn:.4f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 6. Model Comparison

In [ ]:
results_summary = pd.DataFrame({
    'Model':    ['Naive Bayes', 'XGBoost', 'Neural Network'],
    'Val Acc%': [round(acc_nb*100, 2), round(acc_xgb*100, 2), round(acc_nn*100, 2)]
}).sort_values('Val Acc%', ascending=False)

print(results_summary.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.bar(results_summary['Model'], results_summary['Val Acc%'], color=['#4C72B0','#DD8452','#55A868'])
plt.title('Model Validation Accuracy Comparison')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 60)
for i, (_, row) in enumerate(results_summary.iterrows()):
    plt.text(i, row['Val Acc%'] + 0.5, f"{row['Val Acc%']}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Predictions on Test Set

In [ ]:
# Inverse mapping: numeric label → Stay string
inv_stay_map = {v: k for k, v in stay_map.items()}

def decode_stay(arr):
    return pd.Series(arr).map(inv_stay_map)

In [ ]:
# ----- Naive Bayes predictions -----
pred_nb = nb_model.predict(test1.iloc[:, 1:].values)
result_nb = pd.DataFrame({'case_id': test1['case_id'].values,
                           'Stay': decode_stay(pred_nb)})
print('Naive Bayes predictions:')
print(result_nb.head())

In [ ]:
# ----- XGBoost predictions -----
pred_xgb = xgb_model.predict(test1.iloc[:, 1:])
result_xgb = pd.DataFrame({'case_id': test1['case_id'].values,
                             'Stay': decode_stay(pred_xgb)})
print('XGBoost predictions:')
print(result_xgb.head())

In [ ]:
# ----- Neural Network predictions -----
z = test.drop(DROP_TRAIN + ['Stay'], axis=1, errors='ignore')
test_sc = scaler.transform(z)

# Fixed: use np.argmax instead of deprecated predict_classes
pred_nn = np.argmax(model.predict(test_sc), axis=-1)

result_nn = pd.DataFrame({'case_id': test['case_id'].values,
                           'Stay': decode_stay(pred_nn)})
print('Neural Network predictions:')
print(result_nn.head())

---
## 8. Results Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
model_results = [('Naive Bayes', result_nb), ('XGBoost', result_xgb), ('Neural Network', result_nn)]

for ax, (name, res) in zip(axes, model_results):
    counts = res['Stay'].value_counts().reindex(stay_order)
    counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(name)
    ax.set_xlabel('Stay (days)')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Predicted Stay Distribution per Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution tables
for name, res in [('Naive Bayes', result_nb), ('XGBoost', result_xgb), ('Neural Network', result_nn)]:
    print(f'=== {name} ===')
    print(res.groupby('Stay')['case_id'].nunique().reindex(stay_order))
    print()

In [ ]:
# Save best model predictions to CSV
best_model_name  = results_summary.iloc[0]['Model']
best_result_map  = {'Naive Bayes': result_nb, 'XGBoost': result_xgb, 'Neural Network': result_nn}
best_result      = best_result_map[best_model_name]

best_result.to_csv('submission.csv', index=False)
print(f'Saved predictions from best model ({best_model_name}) → submission.csv')